In [3]:
import pandas as pd
import networkx as nx
from networkx.algorithms import bipartite

In [4]:
cast = pd.read_csv(
    "data/processed/production_cast.csv"
)

cast.head()

,performer_id,performer_name,character,production_id
0,473388,Dot Campbell,Chorus,10336
1,34408,Raymond Campbell,Skinny,10336
2,473389,Alice Carter,Chorus,10336
3,35813,Louis Cole,Jimmy,10336
4,438924,Craddock and Shadney,Specialty,10336


In [5]:
print(f"Rows: {len(cast):,}")
print(f"Unique productions: {cast['production_id'].nunique():,}")
print(f"Unique performers: {cast['performer_id'].nunique():,}")

cast.info()

Rows: 115,677
Unique productions: 7,058
Unique performers: 55,263
<class 'pandas.DataFrame'>
RangeIndex: 115677 entries, 0 to 115676
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   performer_id    115677 non-null  int64
 1   performer_name  115677 non-null  str  
 2   character       109958 non-null  str  
 3   production_id   115677 non-null  int64
dtypes: int64(2), str(2)
memory usage: 3.5 MB


In [6]:
cast.isna().sum()

performer_id         0
performer_name       0
character         5719
production_id        0
dtype: int64

In [7]:
duplicate_count = cast.duplicated(
    subset=[
        "production_id",
        "performer_id"
    ]
).sum()

print(f"Duplicate performer-production rows: {duplicate_count:,}")

Duplicate performer-production rows: 38


In [8]:
cast[
    cast.duplicated(
        subset=["production_id", "performer_id"],
        keep=False
    )
].sort_values(
    ["production_id", "performer_id"]
)

,performer_id,performer_name,character,production_id
58064,1219,Ray Harrison,The Great Lover,1614
58073,1219,Ray Harrison,Dance Ensemble,1614
59696,117147,Jacqueline Dalya,Giselle Roublard,1749
59701,117147,Jacqueline Dalya,Giselle Roublard,1749
60009,68634,Edward H. Robins,James Gordon Bennett,1765
...,...,...,...,...
109814,390228,Leah Horowitz,Ensemble,490537
110084,489685,Rachel Potter,Mistress,491036
110097,489685,Rachel Potter,Ensemble,491036
115338,540782,Travis Mitchell,Maxie Dean,543150


In [9]:
cast = cast.drop_duplicates(
    subset=[
        "production_id",
        "performer_id"
    ]
).reset_index(drop=True)

print(f"Rows after deduplication: {len(cast):,}")

Rows after deduplication: 115,639


In [10]:
cast["production_node"] = "P_" + cast["production_id"].astype(str)
cast["performer_node"] = "A_" + cast["performer_id"].astype(str)

cast.head()

,performer_id,performer_name,character,production_id,production_node,performer_node
0,473388,Dot Campbell,Chorus,10336,P_10336,A_473388
1,34408,Raymond Campbell,Skinny,10336,P_10336,A_34408
2,473389,Alice Carter,Chorus,10336,P_10336,A_473389
3,35813,Louis Cole,Jimmy,10336,P_10336,A_35813
4,438924,Craddock and Shadney,Specialty,10336,P_10336,A_438924


In [11]:
import networkx as nx

B = nx.Graph()

# Add performer nodes
performers = cast["performer_node"].unique()

B.add_nodes_from(
    performers,
    bipartite="performer"
)

# Add production nodes
productions = cast["production_node"].unique()

B.add_nodes_from(
    productions,
    bipartite="production"
)

# Add edges
B.add_edges_from(
    cast[
        [
            "performer_node",
            "production_node"
        ]
    ].itertuples(index=False, name=None)
)

print(B)

Graph with 62321 nodes and 115639 edges


In [12]:
from networkx.algorithms import bipartite

performer_nodes = {
    n for n, d in B.nodes(data=True)
    if d["bipartite"] == "performer"
}

production_nodes = {
    n for n, d in B.nodes(data=True)
    if d["bipartite"] == "production"
}

print(
    "Performers:",
    len(performer_nodes)
)

print(
    "Productions:",
    len(production_nodes)
)

Performers: 55263
Productions: 7058


In [13]:
assert all(
    B.nodes[n]["bipartite"] == "performer"
    for n in performer_nodes
)

assert all(
    B.nodes[n]["bipartite"] == "production"
    for n in production_nodes
)

In [14]:
performer_network = bipartite.weighted_projected_graph(
    B,
    performer_nodes
)

print(performer_network)

Graph with 55263 nodes and 1566284 edges


In [15]:
weights = [
    data["weight"]
    for _, _, data in performer_network.edges(data=True)
]

min(weights), max(weights)

(1, 31)

In [16]:
performer_nodes_df = pd.DataFrame(
    {
        "performer_node": list(performer_network.nodes())
    }
)

performer_nodes_df.to_csv(
    "data/processed/performer_nodes.csv",
    index=False
)

In [17]:
edges = nx.to_pandas_edgelist(
    performer_network
)

edges.to_csv(
    "data/processed/performer_edges.csv",
    index=False
)

In [18]:
nx.write_graphml(
    performer_network,
    "data/processed/broadway_network.graphml"
)

In [19]:
weights = [
    data["weight"]
    for _, _, data in performer_network.edges(data=True)
]

import pandas as pd

pd.Series(weights).describe()

count    1.566284e+06
mean     1.050053e+00
std      3.465493e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.100000e+01
dtype: float64

In [22]:
degree = dict(performer_network.degree())

top_degree = sorted(
    degree.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

top_degree



[('A_60670', 952),
 ('A_53589', 720),
 ('A_68475', 701),
 ('A_21639', 698),
 ('A_35584', 692),
 ('A_21617', 675),
 ('A_52148', 652),
 ('A_67079', 650),
 ('A_39487', 642),
 ('A_66920', 640),
 ('A_14598', 635),
 ('A_29701', 633),
 ('A_61800', 632),
 ('A_58170', 625),
 ('A_41881', 619),
 ('A_52830', 602),
 ('A_67892', 595),
 ('A_65498', 595),
 ('A_4031', 593),
 ('A_31445', 593)]

In [23]:
cast[
    cast["performer_node"].isin(
        [x[0] for x in top_degree]
    )
][
    [
        "performer_node",
        "performer_name"
    ]
].drop_duplicates()

,performer_node,performer_name
68,A_58170,Wilma Roeloff
198,A_53589,Victor Moore
402,A_68475,Le Roi Operti
487,A_60670,George Spelvin
966,A_65498,Martin Wolfson
1246,A_61800,Ward Tallman
2069,A_14598,Clarence Derwent
2699,A_67079,Robert Chisholm
4385,A_31445,Walter Beck
4558,A_21639,Dennis King


In [25]:
top_edges = sorted(
    performer_network.edges(data=True),
    key=lambda x: x[2]["weight"],
    reverse=True
)[:20]

top_edges

[('A_40135', 'A_67666', {'weight': 31}),
 ('A_34325', 'A_5481', {'weight': 23}),
 ('A_15732', 'A_67230', {'weight': 22}),
 ('A_67230', 'A_31313', {'weight': 22}),
 ('A_67666', 'A_39873', {'weight': 22}),
 ('A_58434', 'A_67230', {'weight': 21}),
 ('A_5481', 'A_68630', {'weight': 21}),
 ('A_15484', 'A_14789', {'weight': 20}),
 ('A_34325', 'A_68630', {'weight': 20}),
 ('A_67308', 'A_68277', {'weight': 19}),
 ('A_67230', 'A_64183', {'weight': 19}),
 ('A_67028', 'A_68277', {'weight': 19}),
 ('A_40135', 'A_39873', {'weight': 19}),
 ('A_34325', 'A_31445', {'weight': 19}),
 ('A_67666', 'A_67660', {'weight': 19}),
 ('A_67666', 'A_57098', {'weight': 19}),
 ('A_67205', 'A_31313', {'weight': 19}),
 ('A_68277', 'A_67345', {'weight': 19}),
 ('A_67308', 'A_67345', {'weight': 18}),
 ('A_58434', 'A_64183', {'weight': 18})]

In [26]:
top_people = set()

for a,b,data in top_edges:
    top_people.add(a)
    top_people.add(b)

cast[
    cast["performer_node"].isin(top_people)
][
    [
        "performer_node",
        "performer_name"
    ]
].drop_duplicates()

,performer_node,performer_name
853,A_58434,Vera Ross
1680,A_34325,Donald Cameron
1688,A_5481,Eva Le Gallienne
1692,A_68630,Leona Roberts
2408,A_15484,Alfred Lunt
3876,A_14789,Lynn Fontanne
4385,A_31445,Walter Beck
9673,A_67230,William Danforth
10337,A_64183,Herbert Waterous
12565,A_31313,Frances Baviello


In [27]:
components = list(
    nx.connected_components(performer_network)
)

len(components), max(map(len, components))

(285, 52876)

In [28]:
cast[
    cast["performer_node"] == "A_60670"
][
    [
        "performer_name",
        "production_id",
        "character"
    ]
].head(20)

,performer_name,production_id,character
487,George Spelvin,10355,A Poor Debtor
1089,George Spelvin,10378,Charley Dill
1615,George Spelvin,7879,First Alderman
2561,George Spelvin,10499,The Mule
5054,George Spelvin,10629,Nine-foot Giant
7014,George Spelvin,10665,Referee
7845,George Spelvin,10733,A Sergeant of Police
10553,George Spelvin,10852,An Officer
14014,George Spelvin,10990,Buddy Holt
14838,George Spelvin,11024,A Laborer


In [29]:
cast[
    cast["performer_node"] == "A_60670"
].shape

(44, 6)